In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [2]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [4]:
v1.shape

(384,)

In [5]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)
dv.shape

(384,)

In [6]:
v1.dot(dv)

np.float32(0.32332397)

In [7]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [8]:
v2.dot(dv)

np.float32(0.019730438)

In [9]:
from ingest import load_faq_data

documents = load_faq_data()

In [11]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [12]:
from tqdm.auto import tqdm

In [13]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [21]:
vectors[1]

array([-1.09955324e-02, -1.10747449e-01, -2.53693648e-02,  1.97100863e-02,
       -8.37622732e-02, -1.55265450e-01,  6.08169213e-02,  3.26980883e-03,
       -1.84232742e-01,  3.61288860e-02, -2.03759018e-02,  8.37015267e-03,
        8.53649452e-02, -8.11423361e-02, -3.44117843e-02,  2.99429279e-02,
       -4.36164998e-02, -4.33530137e-02,  4.82881553e-02, -4.70029078e-02,
        1.30925979e-02, -3.58360671e-02,  3.54018286e-02, -2.82692146e-02,
        3.28568108e-02,  4.10358943e-02,  9.07346755e-02,  5.90873659e-02,
        1.11789471e-02,  1.03822192e-02, -8.85610003e-03,  2.35163365e-02,
        1.38390986e-02,  5.80496527e-02,  2.97545847e-02, -1.17032062e-02,
        1.00628659e-01, -1.04431011e-01, -9.16769430e-02,  1.54857412e-02,
       -7.07577262e-03,  3.44297290e-02,  1.17166108e-02, -2.63242405e-02,
        5.18442551e-03,  1.29876044e-02, -9.26765241e-03, -5.60835749e-02,
        1.92830581e-02, -9.90842283e-03, -5.16266674e-02, -5.70740327e-02,
       -8.25722069e-02, -

In [15]:
import numpy as np
X = np.array(vectors)
X.shape

(1350, 384)

In [16]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

In [20]:
scores = X.dot(v_query)
scores

array([ 0.48740575,  0.20991933,  0.762941  , ..., -0.08637968,
        0.03759793, -0.03037044], shape=(1350,), dtype=float32)

In [24]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.762941))

In [25]:
documents[idx]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

In [26]:
top5 = np.argsort(scores)[-5:]

In [27]:
top5 = top5[::-1]
top5

array([  2, 625, 907, 538,   7])

In [28]:
scores[top5]

array([0.762941  , 0.7579371 , 0.7192132 , 0.6536312 , 0.56009996],
      dtype=float32)

In [29]:
top5 = np.argsort(-scores)[:5]

In [30]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.762941
{'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

0.7579371
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}

0.7192132
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related 

In [31]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [32]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

In [33]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '41aabbd7c5',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'},
 {'id': '2d8b16c2a0',
  'course': 'mlops-zoomcamp',
  'section':

In [36]:
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [37]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the co